# Fine-tuning QLoRA — Clasificador de riesgo IA (EU AI Act)

> **⚠ Este notebook está diseñado para ejecutarse en Google Colab con GPU T4 (16 GB VRAM).**

Pasos:
0. Instalación de dependencias (solo Colab)
1. Configuración del modelo base y QLoRA
2. Carga del dataset
3. Tokenización
4. Fine-tuning con SFTTrainer
5. Evaluación en test
6. Guardar adaptador LoRA
7. Registro en MLflow

## 0. Instalación (solo Colab)

In [1]:
import os, sys, subprocess

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# ── PyTorch ──────────────────────────────────────────────────────────────────
try:
    import torch
    print(f"✓ torch {torch.__version__} ya instalado")
except ImportError:
    print("Instalando PyTorch (CPU por defecto)…")
    print("  Para GPU con CUDA visita: https://pytorch.org/get-started/locally/")
    _pip("torch", "torchvision", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/cpu")
    import torch

# ── Resto de dependencias ─────────────────────────────────────────────────────
_missing = [pkg for pkg in ["transformers", "peft", "trl", "datasets", "accelerate"]
            if __import__("importlib").util.find_spec(pkg) is None]
if _missing:
    print(f"Instalando: {', '.join(_missing)}…")
    _pip("transformers==4.47.1", "peft==0.14.0",
         "trl==0.13.0", "datasets==3.2.0", "accelerate==1.2.1")

# ── bitsandbytes (cuantización 4-bit, requiere GPU NVIDIA) ───────────────────
try:
    import bitsandbytes  # noqa: F401
    _BNB_OK = True
except ImportError:
    try:
        print("Instalando bitsandbytes…")
        _pip("bitsandbytes==0.45.0")
        import bitsandbytes  # noqa: F401
        _BNB_OK = True
    except Exception as _e:
        print(f"⚠ bitsandbytes no disponible ({_e}).")
        print("  La cuantización 4-bit estará deshabilitada (se requiere GPU NVIDIA).")
        _BNB_OK = False

print("✓ Dependencias listas")

Instalando PyTorch (CPU por defecto)…
  Para GPU con CUDA visita: https://pytorch.org/get-started/locally/


The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


✓ Dependencias listas


## 1. Configuración

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# ── Detección de hardware ─────────────────────────────────────────────────────
_CUDA = torch.cuda.is_available()
_BNB_OK = globals().get("_BNB_OK", False)   # definido en la celda de instalación
USE_QUANTIZATION = _CUDA and _BNB_OK         # QLoRA 4-bit solo con GPU + bitsandbytes

# ── Selección de modelo según hardware ────────────────────────────────────────
if _CUDA:
    # GPU: modelo 7B completo con QLoRA (recomendado, ~8 GB VRAM)
    BASE_MODEL = "mistralai/Mistral-7B-v0.1"
else:
    # CPU: modelo ligero sin cuantización (solo para pruebas, entrenamiento lento)
    BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    print("⚠  Sin GPU detectada: se usará TinyLlama-1.1B en CPU.")
    print("   El fine-tuning será muy lento. Para resultados reales usa una GPU o Colab.")

# ── Configuración QLoRA 4-bit (solo si GPU + bitsandbytes disponibles) ────────
bnb_config = None
if USE_QUANTIZATION:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

# ── Configuración LoRA ────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

print(f"Modelo base:        {BASE_MODEL}")
print(f"Dispositivo:        {'CUDA (' + torch.cuda.get_device_name(0) + ')' if _CUDA else 'CPU'}")
print(f"Cuantización 4-bit: {'Sí (QLoRA)' if USE_QUANTIZATION else 'No'}")
if _CUDA:
    print(f"VRAM disponible:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠  Sin GPU detectada: se usará TinyLlama-1.1B en CPU.
   El fine-tuning será muy lento. Para resultados reales usa una GPU o Colab.
Modelo base:        TinyLlama/TinyLlama-1.1B-Chat-v1.0
Dispositivo:        CPU
Cuantización 4-bit: No


In [3]:
# Cargar tokenizador y modelo
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

load_kwargs = dict(trust_remote_code=True)
if USE_QUANTIZATION:
    # GPU con bitsandbytes: cuantización 4-bit, reparto automático por capas
    load_kwargs["quantization_config"] = bnb_config
    load_kwargs["device_map"] = "auto"
else:
    # CPU o GPU sin bitsandbytes: precisión completa (float32)
    load_kwargs["torch_dtype"] = torch.float32
    load_kwargs["device_map"] = "cpu" if not _CUDA else "auto"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kwargs)
model.config.use_cache = False

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rammu\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling b

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


## 2. Carga del dataset

In [4]:
from datasets import load_dataset

# Rutas locales (ajustar si se ejecuta en Colab con Drive montado)
PATH_TRAIN = "data/finetune/train.jsonl"
PATH_TEST  = "data/finetune/test.jsonl"

dataset = load_dataset(
    "json",
    data_files={"train": PATH_TRAIN, "test": PATH_TEST},
)

print(dataset)
print()
print("Ejemplo de muestra:")
print(dataset["train"][0]["text"])

Generating train split: 240 examples [00:00, 9219.35 examples/s]
Generating test split: 60 examples [00:00, 7490.50 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'etiqueta'],
        num_rows: 240
    })
    test: Dataset({
        features: ['text', 'etiqueta'],
        num_rows: 60
    })
})

Ejemplo de muestra:
### Instrucción:
Clasifica el siguiente sistema de IA según el Reglamento de Inteligencia Artificial de la UE en una de estas categorías: inaceptable, alto_riesgo, riesgo_limitado, riesgo_minimo.

### Descripción:
Un sistema de inteligencia artificial de ciudad inteligente emite automáticamente multas de tránsito mediante el reconocimiento de matrículas sin permitir la revisión o explicación humana de sus decisiones.

### Clasificación:
alto_riesgo


## 3. Tokenización

In [5]:
MAX_LENGTH = 512

def tokenizar(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

dataset_tokenizado = dataset.map(tokenizar, batched=True)

print(f"Train tokenizado: {len(dataset_tokenizado['train'])} muestras")
print(f"Test tokenizado:  {len(dataset_tokenizado['test'])} muestras")

Map: 100%|██████████| 60/60 [00:00<00:00, 1903.11 examples/s]

Train tokenizado: 240 muestras
Test tokenizado:  60 muestras


## 4. Fine-tuning con SFTTrainer

In [6]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "model/qlora_checkpoints"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4 if _CUDA else 1,   # batch pequeño en CPU
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    fp16=_CUDA and USE_QUANTIZATION,   # fp16 solo en GPU con QLoRA
    optim="paged_adamw_8bit" if USE_QUANTIZATION else "adamw_torch",
    max_seq_length=MAX_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    peft_config=lora_config,
)

print("Iniciando fine-tuning…")
train_result = trainer.train()

train_loss = train_result.training_loss
print(f"\n✓ Fine-tuning completado")
print(f"  Train loss final: {train_loss:.4f}")

Map: 100%|██████████| 240/240 [00:00<00:00, 6951.02 examples/s]


Iniciando fine-tuning…


  0%|          | 0/180 [00:00<?, ?it/s]c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  6%|▌         | 10/180 [01:26<23:09,  8.18s/it]

{'loss': 2.2609, 'grad_norm': 1.6398141384124756, 'learning_rate': 0.000199739323151795, 'epoch': 0.17}


 11%|█         | 20/180 [02:47<21:59,  8.25s/it]

...

{'loss': 0.7025, 'grad_norm': 1.514378309249878, 'learning_rate': 1.6255156067997323e-06, 'epoch': 2.83}


100%|██████████| 180/180 [25:55<00:00,  8.78s/it]

{'loss': 0.691, 'grad_norm': 1.540365219116211, 'learning_rate': 0.0, 'epoch': 3.0}


100%|██████████| 180/180 [25:56<00:00,  8.64s/it]

{'train_runtime': 1556.087, 'train_samples_per_second': 0.463, 'train_steps_per_second': 0.116, 'train_loss': 0.9155389149983724, 'epoch': 3.0}

✓ Fine-tuning completado
  Train loss final: 0.9155


## 5. Evaluación en test

In [7]:
import re
import numpy as np
from sklearn.metrics import classification_report, f1_score

CLASES = ["inaceptable", "alto_riesgo", "riesgo_limitado", "riesgo_minimo"]

def extraer_etiqueta(texto_generado):
    """Extrae la etiqueta de la respuesta generada por el modelo."""
    texto = texto_generado.lower()
    for clase in CLASES:
        if clase in texto:
            return clase
    return "desconocido"

def predecir(texto_entrada):
    """Genera una predicción para una muestra."""
    # El prompt es todo hasta '### Clasificación:\n'
    prompt = texto_entrada.split("### Clasificación:")[0] + "### Clasificación:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
        )
    respuesta = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return extraer_etiqueta(respuesta)

print("Evaluando en test...")
y_true = [row["etiqueta"] for row in dataset["test"]]
y_pred = [predecir(row["text"]) for row in dataset["test"]]

print("\n=== Resultados en TEST (QLoRA) ===")
print(classification_report(y_true, y_pred))

finetune_f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
finetune_accuracy = np.mean([p == t for p, t in zip(y_pred, y_true)])

print(f"F1-macro: {finetune_f1_macro:.4f}")
print(f"Accuracy: {finetune_accuracy:.4f}")

c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\transformers\generation\configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Evaluando en test...

=== Resultados en TEST (QLoRA) ===
                 precision    recall  f1-score   support

    alto_riesgo       0.00      0.00      0.00         9
    inaceptable       0.69      0.96      0.81        26
riesgo_limitado       0.00      0.00      0.00         4
  riesgo_minimo       0.88      1.00      0.93        21

       accuracy                           0.77        60
      macro avg       0.39      0.49      0.43        60
   weighted avg       0.61      0.77      0.68        60

F1-macro: 0.4349
Accuracy: 0.7667


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

## 6. Guardar adaptador LoRA

In [8]:
import os

ADAPTER_DIR = "model/qlora_adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"✓ Adaptador LoRA guardado en: {ADAPTER_DIR}")
print(f"  Archivos: {os.listdir(ADAPTER_DIR)}")

✓ Adaptador LoRA guardado en: model/qlora_adapter
  Archivos: ['adapter_config.json', 'adapter_model.safetensors', 'README.md', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


## 7. Registro en MLflow

In [9]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
import mlflow

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "")
MLFLOW_EXPERIMENT   = "clasificador_riesgo_ia"

try:
    if MLFLOW_TRACKING_URI:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="finetune_qlora"):
        # Parámetros
        mlflow.log_params({
            "base_model":                  BASE_MODEL,
            "lora_r":                      lora_config.r,
            "lora_alpha":                  lora_config.lora_alpha,
            "lora_dropout":                lora_config.lora_dropout,
            "lora_target_modules":         str(lora_config.target_modules),
            "bnb_4bit_quant_type":         "nf4",
            "bnb_use_double_quant":        True,
            "num_train_epochs":            training_args.num_train_epochs,
            "per_device_train_batch_size": training_args.per_device_train_batch_size,
            "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
            "learning_rate":               training_args.learning_rate,
            "lr_scheduler_type":           training_args.lr_scheduler_type,
            "max_seq_length":              MAX_LENGTH,
        })

        # Métricas
        mlflow.log_metrics({
            "train_loss":         train_loss,
            "finetune_f1_macro":  finetune_f1_macro,
            "finetune_accuracy":  finetune_accuracy,
        })

        # Artefactos
        mlflow.log_artifacts(ADAPTER_DIR, artifact_path="qlora_adapter")

        print(f"✓ Fine-tuning QLoRA registrado en MLflow")
        print(f"  F1-macro: {finetune_f1_macro:.4f}")
        print(f"  Run ID:   {mlflow.active_run().info.run_id}")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

2026/02/21 01:27:43 INFO mlflow.tracking.fluent: Experiment with name 'clasificador_riesgo_ia' does not exist. Creating a new experiment.


✓ Fine-tuning QLoRA registrado en MLflow
  F1-macro: 0.4349
  Run ID:   20bc01a20b8e4c47b87369cba5030b5b
